In [ ]:
import numpy as np
from scipy.io import loadmat
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
import torch

In [ ]:
save_path = {
    "train": "trainset.h5",
    "test": "testset.h5",
}

In [ ]:
def resize_batch_numpy(data, target_size=64, is_binary=False):
    """
    data: numpy array with shape [B, 1, H, W]
    target_size: int (e.g., 64)
    is_binary: True -> nearest; False -> bilinear
    """
    B, C, H, W = data.shape
    scale = target_size / H
    order = 0 if is_binary else 1  # 0=nearest, 1=bilinear

    data_resized = zoom(data, (1, 1, scale, scale), order=order)

    return data_resized

In [ ]:
def minmax_norm(x, xmin, xmax):
    """Min-Max to [-1, 1]"""
    return 2 * (x - xmin) / (xmax - xmin) - 1

In [ ]:
def obs_field_reconstruction(obs_batch, field_size):
    B, N, T = obs_batch.shape
    width = T - 2

    obs_recon_batch = torch.zeros((B, width, field_size, field_size),
                                 device=obs_batch.device)

    x_pos = ((obs_batch[..., 0] + 1) / 2 * field_size).long()  # (B, N)
    y_pos = ((obs_batch[..., 1] + 1) / 2 * field_size).long()  # (B, N)

    for b in range(B):
        for w in range(width):
            obs_recon_batch[b, w, x_pos[b], y_pos[b]] = obs_batch[b, :, w + 2]

    return obs_recon_batch

### Trainset Process

In [ ]:
stress_obs_train = np.load("TrainingDataset/dataset_full_stress.npy")
disp_obs_train = np.load("TrainingDataset/dataset_full_displacement.npy")
target_train = np.load("TrainingDataset/dataset_labels.npy")
stress_obs_train = np.transpose(stress_obs_train, (0, 3, 1, 2))
disp_obs_train = np.transpose(disp_obs_train, (0, 3, 1, 2))

In [ ]:
target_max, target_min = target_train.max(), target_train.min()
target_trainset_norm = minmax_norm(target_train, target_min, target_max)
stress_max, stress_min = stress_obs_train.max(), stress_obs_train.min()
stress_norm = minmax_norm(stress_obs_train, stress_min, stress_max)
disp_max, disp_min = disp_obs_train.max(), disp_obs_train.min()
disp_norm = minmax_norm(disp_obs_train, disp_min, disp_max)
obs_label_train_norm = np.concatenate((stress_norm, disp_norm), axis=1)
target_trainset_norm = np.expand_dims(target_trainset_norm, axis=1)
print(obs_label_train_norm.shape, target_trainset_norm.shape)

In [ ]:
obs_set_train = []

for sample in obs_label_train_norm:
    set_list = []
    for x in range(obs_label_train_norm.shape[-1]):
        for y in range(obs_label_train_norm.shape[-1]):
            element = [minmax_norm(x,0,64),minmax_norm(y,0,64)] + sample[:,x,y].tolist()
            set_list.append(element)
    obs_set_train.append(set_list)
obs_set_train = np.array(obs_set_train)

### Testset Process

In [ ]:
stress_obs_test = np.load("TestingDataset/dataset_full_stress.npy")
disp_obs_test = np.load("TestingDataset/dataset_full_displacement.npy")
target_test = np.load("TestingDataset/dataset_labels.npy")
stress_obs_test = np.transpose(stress_obs_test, (0, 3, 1, 2))
disp_obs_test = np.transpose(disp_obs_test, (0, 3, 1, 2))

print(stress_obs_test.max(),stress_obs_test.min())
print(disp_obs_test.max(),disp_obs_test.min())
print(target_test.max(),target_test.min())

In [ ]:
# target_max, target_min = target_train.max(), target_train.min()
target_test_norm = minmax_norm(target_test, target_min, target_max)
# stress_max, stress_min = stress_obs_train.max(), stress_obs_train.min()
stress_norm_test = minmax_norm(stress_obs_test, stress_min, stress_max)
# disp_max, disp_min = disp_obs_train.max(), disp_obs_train.min()
disp_norm_test = minmax_norm(disp_obs_test, disp_min, disp_max)

obs_label_testset_norm = np.concatenate((stress_norm_test, disp_norm_test), axis=1)
target_test_norm = np.expand_dims(target_test_norm, axis=1)
print(obs_label_testset_norm.shape, target_test_norm.shape)

In [ ]:
obs_set_test = []

for sample in obs_label_testset_norm:
    set_list = []
    for x in range(obs_label_testset_norm.shape[-1]):
        for y in range(obs_label_testset_norm.shape[-1]):
            element = [minmax_norm(x,0,64),minmax_norm(y,0,64)] + sample[:,x,y].tolist()
            set_list.append(element)
    obs_set_test.append(set_list)
obs_set_test = np.array(obs_set_test)

In [ ]:
with h5py.File(f'../{save_path["train"]}', "w") as f:
    f.create_dataset("target", data=target_trainset_norm.squeeze(1))
    f.create_dataset("obs", data=obs_set_train)
    f.create_dataset("obs_label", data=obs_label_train_norm)
print(target_trainset_norm.squeeze(1).shape,obs_set_train.shape,obs_label_train_norm.shape)

In [ ]:
with h5py.File(f'../{save_path["test"]}', "w") as f:
    f.create_dataset("target", data=target_test_norm.squeeze(1))
    f.create_dataset("obs", data=obs_set_test)
    f.create_dataset("obs_label", data=obs_label_testset_norm)
    print(target_test_norm.squeeze(1).shape,obs_set_test.shape,obs_label_testset_norm.shape)